# Proyecto Grafos

## Clases Principales
## Nodo: funciones de creación e identificación
## Grafo: funciones de creación, nodos y aristas

In [3]:
class Nodo:
  def __init__(self, id):
    self.id = id

  def identificar(self):
    return(f"Nodo id={self.id}")

  def comprobar(self, other):
      if not isinstance(other, Nodo):
          return NotImplemented
      return self.id == other.id


class Arista:
  def __init__(self, origen, destino, distancia=1):
    if not isinstance(origen, Nodo) or not isinstance(destino, Nodo):
        raise ValueError("Origen y destino deben ser instancias de Nodo")
    self.origen = origen
    self.destino = destino
    self.distancia = distancia

  def identificar(self):
    return(f"Arista ({self.origen.id},{self.destino.id}) distancia={self.distancia}")

  def comprobar(self, other):
    if not isinstance(other, Arista):
        return NotImplemented
    return (self.origen == other.origen and self.destino == other.destino and self.distancia == other.distancia) or \
           (self.origen == other.destino and self.destino == other.origen and self.distancia == other.distancia)


class Grafo:
    def __init__(self, dirigido=False):
        #Adyacencia es un diccionario que mapea el ID de un nodo a una lista de aristas que salen de él
        self.nodos = set()
        self.aristas = set()
        self.adyacencia = {}
        self.dirigido = dirigido

    def agregar_nodo(self, nodo):
        if not isinstance(nodo, Nodo):
            raise ValueError("El elemento a agregar debe ser una instancia de Nodo")
        existe = any(nodo.comprobar(n) is True for n in self.nodos)
        if existe:
            raise ValueError("El nodo ya existe en el grafo")
        self.nodos.add(nodo)
        self.adyacencia[nodo.id] = []

    def agregar_arista(self, arista):
        if not isinstance(arista, Arista):
            raise ValueError("El elemento a agregar debe ser una instancia de Arista")
        # Buscar si ya existen instancias equivalentes de origen y destino por ID
        origen_existente = self._buscar_nodo_por_id(arista.origen.id)
        destino_existente = self._buscar_nodo_por_id(arista.destino.id)

        if not origen_existente:
            self.agregar_nodo(arista.origen)
            origen_existente = arista.origen
        if not destino_existente:
            self.agregar_nodo(arista.destino)
            destino_existente = arista.destino

        # Usar las referencias existentes
        arista_limpia = Arista(origen_existente, destino_existente, arista.distancia)
        self.aristas.add(arista_limpia)
        self.adyacencia[origen_existente.id].append(arista_limpia)
        #Si el grafo no es dirigido
        if not self.dirigido:
            arista_inversa = Arista(destino_existente, origen_existente, arista.distancia)
            self.adyacencia[destino_existente.id].append(arista_inversa)

    def get_nodos(self):
        return list(self.nodos)

    def get_aristas(self):
        return list(self.aristas)

    #Lista de los nodos a los que se puede llegar directamente desde 'nodo_id'.
    def get_vecinos(self, nodo_id):
        if nodo_id not in self.adyacencia:
            raise ValueError(f"El nodo con ID {nodo_id} no existe en el grafo.")
        return [arista.destino for arista in self.adyacencia[nodo_id]]

    def identificar(self):
        return(f"Grafo(numero de nodos={len(self.nodos)}, numero de aristas={len(self.aristas)}, dirigido={self.dirigido})")

    def guardar_graphviz(self, ruta_archivo):
        conector = "->" if self.dirigido else "--"
        tipo_grafo = "digraph" if self.dirigido else "graph"
        lineas = []
        lineas.append(f"{tipo_grafo} G {{")
        #Nodos
        for nodo in sorted(self.get_nodos(), key=lambda n: n.id):
            lineas.append(f"  {nodo.id};")
        #Aristas
        if self.dirigido:
            for arista in self.get_aristas():
                lineas.append(f"  {arista.origen.id} {conector} {arista.destino.id} [label=\"{arista.distancia}\"];")
        else:
            #Para no duplicar aristas bidireccionales en el formato simple no dirigido
            aristas_guardadas = set()
            for arista in self.get_aristas():
                par = tuple(sorted([arista.origen.id, arista.destino.id]))
                if par not in aristas_guardadas:
                    aristas_guardadas.add(par)
                    lineas.append(f"  {arista.origen.id} {conector} {arista.destino.id} [label=\"{arista.distancia}\"];")
        lineas.append("}")
        with open(ruta_archivo, "w", encoding="utf-8") as f:
            f.write("\n".join(lineas) + "\n")

    def _buscar_nodo_por_id(self, nodo_id):
        for nodo in self.nodos:
            if nodo.id == nodo_id:
                return nodo
        return None

##Modelos de Generación de Grafos

In [4]:
import random
import math


"""
Modelos de malla
m: número de columnas (> 1)
n: número de filas (> 1)
"""
def grafoMalla(m, n, dirigido=False):
    if m <= 1 or n <= 1:
        raise ValueError("m y n deben ser mayores que 1")
    grafo = Grafo(dirigido=dirigido)
    #Matriz de nodos para facilitar el acceso por coordenadas (i, j)
    nodos = {}
    for i in range(m):
        for j in range(n):
            #Identificador único para cada nodo en la malla
            nodo_id = f"n_{i}_{j}"
            nodo = Nodo(nodo_id)
            grafo.agregar_nodo(nodo)
            nodos[(i, j)] = nodo
    #Aristas según el modelo de malla
    for i in range(m):
        for j in range(n):
            nodo_actual = nodos[(i, j)]
            #Conectar con el nodo de la siguiente columna (i+1, j)
            if i < m - 1:
                nodo_derecha = nodos[(i + 1, j)]
                grafo.agregar_arista(Arista(nodo_actual, nodo_derecha))
            #Conectar con el nodo de la siguiente fila (i, j+1)
            if j < n - 1:
                nodo_abajo = nodos[(i, j + 1)]
                grafo.agregar_arista(Arista(nodo_actual, nodo_abajo))
    return grafo


"""
Modelo de Erdős-Rényi (G_{n,M}).
n: Número de nodos (> 0)
m: Número de aristas a crear (0 <= m <= n*(n-1)/2 si no dirigido)
"""
def grafoErdosRenyi(n, m, dirigido=False):
    if n <= 0:
        raise ValueError("El número de nodos debe ser mayor a 0")
    grafo = Grafo(dirigido=dirigido)
    nodos = [Nodo(i) for i in range(n)]
    for nodo in nodos:
        grafo.agregar_nodo(nodo)
    #Generar todos los pares posibles de conexiones únicas
    posibles_aristas = []
    for i in range(n):
        for j in range(n):
            if i != j:
                if dirigido or i < j:
                    posibles_aristas.append((nodos[i], nodos[j]))
    if m > len(posibles_aristas):
        raise ValueError(f"Demasiadas aristas solicitadas. El máximo posible para n={n} es {len(posibles_aristas)}")
    #Seleccionar aleatoriamente m aristas
    aristas_seleccionadas = random.sample(posibles_aristas, m)
    for origen, destino in aristas_seleccionadas:
        grafo.agregar_arista(Arista(origen, destino))
    return grafo


"""
Modelo de Gilbert (G_{n,p}).
n: Número de nodos (> 0)
p: Probabilidad de creación de cada arista (0.0 <= p <= 1.0)
"""
def grafoGilbert(n, p, dirigido=False):
    if n <= 0:
        raise ValueError("El número de nodos debe ser mayor a 0")
    if not (0.0 <= p <= 1.0):
        raise ValueError("La probabilidad p debe estar entre 0.0 y 1.0")
    grafo = Grafo(dirigido=dirigido)
    nodos = [Nodo(i) for i in range(n)]
    for nodo in nodos:
        grafo.agregar_nodo(nodo)
    #Recorrer todos los pares y agregarlos basándose en la probabilidad p
    for i in range(n):
        for j in range(n):
            if i != j:
                if dirigido or i < j:
                    if random.random() < p:
                        grafo.agregar_arista(Arista(nodos[i], nodos[j]))
    return grafo


"""
Modelo geográfico simple
n: número de nodos (> 0)
r: distancia máxima para crear una arista (0, 1)
"""
def grafoGeografico(n, r, dirigido=False):
    if n <= 0:
        raise ValueError("El número de nodos debe ser mayor a 0")
    if not (0 <= r <= 1):
        raise ValueError("La distancia r debe estar entre 0 y 1")
    grafo = Grafo(dirigido=dirigido)
    nodos = [Nodo(i) for i in range(n)]
    #Coordenadas aleatorias (x, y) en el plano
    coordenadas = {nodo.id: (random.random(), random.random()) for nodo in nodos}
    for nodo in nodos:
        grafo.agregar_nodo(nodo)
    #Crear aristas si la distancia euclidiana es menor o igual a r
    for i in range(n):
        for j in range(n):
            if i != j:
                if dirigido or i < j:
                    x1, y1 = coordenadas[nodos[i].id]
                    x2, y2 = coordenadas[nodos[j].id]
                    distancia = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
                    if distancia <= r:
                        grafo.agregar_arista(Arista(nodos[i], nodos[j]))
    return grafo

"""
Modelo Barabasi-Albert
n: número de nodos (> 0)
d: grado máximo esperado por cada nodo (> 1)
"""
def grafoBarabasiAlbert(n, d, dirigido=False):
    if n <= 0:
        raise ValueError("El número de nodos debe ser mayor a 0")
    if d <= 1 or d >= n:
        raise ValueError("d debe ser mayor que 1 y menor que n")
    grafo = Grafo(dirigido=dirigido)
    nodos = [Nodo(i) for i in range(n)]
    #Los primeros d nodos se conectan todos con todos (grafo completo inicial)
    for i in range(d):
        grafo.agregar_nodo(nodos[i])
    for i in range(d):
        for j in range(i + 1, d):
            grafo.agregar_arista(Arista(nodos[i], nodos[j]))
    #Agregar los nodos restantes uno a uno mediante conexión preferencial
    for i in range(d, n):
        nuevo_nodo = nodos[i]
        grafo.agregar_nodo(nuevo_nodo)
        #Calcular grados de todos los nodos actualmente en el grafo
        nodos_existentes = grafo.get_nodos()
        grados = {}
        for n_ext in nodos_existentes:
            if n_ext.id == nuevo_nodo.id:
                continue
            #Grado de un nodo se puede determinar sumando sus conexiones en la lista de adyacencia
            grados[n_ext] = len(grafo.adyacencia.get(n_ext.id, []))
        total_grados = sum(grados.values())
        #Seleccionar d destinos basados en la probabilidad proporcional a su grado
        destinos_seleccionados = []
        intentos = 0
        while len(destinos_seleccionados) < d and intentos < 100:
            intentos += 1
            if total_grados == 0:
                #Si no hay aristas aún, seleccionamos uniformemente al azar
                candidato = random.choice([nd for nd in nodos_existentes if nd.id != nuevo_nodo.id])
            else:
                r_val = random.uniform(0, total_grados)
                acumulado = 0
                candidato = None
                for nd, g in grados.items():
                    acumulado += g
                    if r_val <= acumulado:
                        candidato = nd
                        break
            if candidato and candidato not in destinos_seleccionados:
                destinos_seleccionados.append(candidato)
        for destino in destinos_seleccionados:
            grafo.agregar_arista(Arista(nuevo_nodo, destino))
    return grafo


"""
Modelo Dorogovtsev-Mendes
n: número de nodos (≥ 3)
"""
def grafoDorogovtsevMendes(n, dirigido=False):
    if n < 3:
        raise ValueError("El número de nodos debe ser mayor o igual a 3")
    grafo = Grafo(dirigido=dirigido)
    nodos = [Nodo(i) for i in range(n)]
    #Crear el triángulo inicial con los primeros 3 nodos
    for i in range(3):
        grafo.agregar_nodo(nodos[i])
    grafo.agregar_arista(Arista(nodos[0], nodos[1]))
    grafo.agregar_arista(Arista(nodos[1], nodos[2]))
    grafo.agregar_arista(Arista(nodos[2], nodos[0]))
    #Agregar los nodos restantes seleccionando una arista al azar
    for i in range(3, n):
        nuevo_nodo = nodos[i]
        grafo.agregar_nodo(nuevo_nodo)
        #Elegir una arista al azar del conjunto de aristas registradas
        arista_azar = random.choice(list(grafo.get_aristas()))
        #Conectar el nuevo nodo a ambos extremos de la arista elegida
        grafo.agregar_arista(Arista(nuevo_nodo, arista_azar.origen))
        grafo.agregar_arista(Arista(nuevo_nodo, arista_azar.destino))
    return grafo

Generar grafos

In [ ]:
# Generación de 3 grafos (50, 200 y 500 nodos) por cada modelo

print("Generando grafos del modelo: Malla")
# 50 nodos (5x10), 200 nodos (10x20), 500 nodos (20x25)
malla_50 = grafoMalla(5, 10)
malla_50.guardar_graphviz("malla_50.gv")

malla_200 = grafoMalla(10, 20)
malla_200.guardar_graphviz("malla_200.gv")

malla_500 = grafoMalla(20, 25)
malla_500.guardar_graphviz("malla_500.gv")
print(" -> Guardados malla_50.gv, malla_200.gv, malla_500.gv")

print("Generando grafos del modelo: Erdős-Rényi")
erdos_renyi_50 = grafoErdosRenyi(50, 100)
erdos_renyi_50.guardar_graphviz("erdos_renyi_50.gv")

erdos_renyi_200 = grafoErdosRenyi(200, 400)
erdos_renyi_200.guardar_graphviz("erdos_renyi_200.gv")

erdos_renyi_500 = grafoErdosRenyi(500, 1000)
erdos_renyi_500.guardar_graphviz("erdos_renyi_500.gv")
print(" -> Guardados erdos_renyi_50.gv, erdos_renyi_200.gv, erdos_renyi_500.gv")

print("Generando grafos del modelo: Gilbert")
gilbert_50 = grafoGilbert(50, 0.08)
gilbert_50.guardar_graphviz("gilbert_50.gv")

gilbert_200 = grafoGilbert(200, 0.02)
gilbert_200.guardar_graphviz("gilbert_200.gv")

gilbert_500 = grafoGilbert(500, 0.008)
gilbert_500.guardar_graphviz("gilbert_500.gv")
print(" -> Guardados gilbert_50.gv, gilbert_200.gv, gilbert_500.gv")

print("Generando grafos del modelo: Geográfico")
geografico_50 = grafoGeografico(50, 0.2)
geografico_50.guardar_graphviz("geografico_50.gv")

geografico_200 = grafoGeografico(200, 0.1)
geografico_200.guardar_graphviz("geografico_200.gv")

geografico_500 = grafoGeografico(500, 0.06)
geografico_500.guardar_graphviz("geografico_500.gv")
print(" -> Guardados geografico_50.gv, geografico_200.gv, geografico_500.gv")

print("Generando grafos del modelo: Barabási-Albert")
barabasi_albert_50 = grafoBarabasiAlbert(50, 3)
barabasi_albert_50.guardar_graphviz("barabasi_albert_50.gv")

barabasi_albert_200 = grafoBarabasiAlbert(200, 3)
barabasi_albert_200.guardar_graphviz("barabasi_albert_200.gv")

barabasi_albert_500 = grafoBarabasiAlbert(500, 3)
barabasi_albert_500.guardar_graphviz("barabasi_albert_500.gv")
print(" -> Guardados barabasi_albert_50.gv, barabasi_albert_200.gv, barabasi_albert_500.gv")

print("Generando grafos del modelo: Dorogovtsev-Mendes")
dorogovtsev_mendes_50 = grafoDorogovtsevMendes(50)
dorogovtsev_mendes_50.guardar_graphviz("dorogovtsev_mendes_50.gv")

dorogovtsev_mendes_200 = grafoDorogovtsevMendes(200)
dorogovtsev_mendes_200.guardar_graphviz("dorogovtsev_mendes_200.gv")

dorogovtsev_mendes_500 = grafoDorogovtsevMendes(500)
dorogovtsev_mendes_500.guardar_graphviz("dorogovtsev_mendes_500.gv")
print(" -> Guardados dorogovtsev_mendes_50.gv, dorogovtsev_mendes_200.gv, dorogovtsev_mendes_500.gv")

Generando grafos del modelo: Malla
 -> Guardados malla_50.gv, malla_200.gv, malla_500.gv
Generando grafos del modelo: Erdős-Rényi
 -> Guardados erdos_renyi_50.gv, erdos_renyi_200.gv, erdos_renyi_500.gv
Generando grafos del modelo: Gilbert
 -> Guardados gilbert_50.gv, gilbert_200.gv, gilbert_500.gv
Generando grafos del modelo: Geográfico
 -> Guardados geografico_50.gv, geografico_200.gv, geografico_500.gv
Generando grafos del modelo: Barabási-Albert
 -> Guardados barabasi_albert_50.gv, barabasi_albert_200.gv, barabasi_albert_500.gv
Generando grafos del modelo: Dorogovtsev-Mendes
 -> Guardados dorogovtsev_mendes_50.gv, dorogovtsev_mendes_200.gv, dorogovtsev_mendes_500.gv


##Algoritmos BFS y DFS

In [5]:
"""
BFS - Busqueda a lo Ancho
"""
def BFS(grafo, s):
    # s es el ID del nodo origen
    arbol = Grafo(dirigido=True)
    origen = grafo._buscar_nodo_por_id(s)
    if not origen:
        raise ValueError(f"El nodo origen {s} no existe en el grafo.")
    visitados = {s}
    arbol.agregar_nodo(Nodo(s))
    cola = [origen]
    while cola:
        actual = cola.pop(0)
        # Buscamos las aristas que salen del nodo actual
        for arista in grafo.adyacencia.get(actual.id, []):
            vecino = arista.destino
            if vecino.id not in visitados:
                visitados.add(vecino.id)
                # Añadir arista de forma limpia delegando la adición de nodos a agregar_arista
                arbol.agregar_arista(Arista(Nodo(actual.id), Nodo(vecino.id), arista.distancia))
                cola.append(vecino)
    return arbol


"""
DFS - Busqueda a Profundidad
Algoritmo Recursivo
"""
def DFS_R(grafo, s):
    # s es el ID del nodo origen
    arbol = Grafo(dirigido=True)
    origen = grafo._buscar_nodo_por_id(s)
    if not origen:
        raise ValueError(f"El nodo origen {s} no existe en el grafo.")
    visitados = set()
    def dfs_recursivo(u_id):
        visitados.add(u_id)
        if len(arbol.nodos) == 0:
            arbol.agregar_nodo(Nodo(u_id))
        for arista in grafo.adyacencia.get(u_id, []):
            v_id = arista.destino.id
            if v_id not in visitados:
                arbol.agregar_arista(Arista(Nodo(u_id), Nodo(v_id), arista.distancia))
                dfs_recursivo(v_id)
    dfs_recursivo(s)
    return arbol


"""
DFS - Busqueda a Profundidad
Algoritmo Iterativo
"""
def DFS_I(grafo, s):
    # s es el ID del nodo origen
    arbol = Grafo(dirigido=True)
    origen = grafo._buscar_nodo_por_id(s)
    if not origen:
        raise ValueError(f"El nodo origen {s} no existe en el grafo.")
    visitados = set()
    pila = [origen]
    padre = {s: None} # Mapea ID del nodo hijo a (ID del nodo padre, distancia)
    while pila:
        actual = pila.pop()
        if actual.id not in visitados:
            visitados.add(actual.id)
            # Al visitar un nodo, lo agregamos al árbol junto con su arista desde el padre
            if padre[actual.id] is not None:
                id_padre, dist = padre[actual.id]
                arbol.agregar_arista(Arista(Nodo(id_padre), Nodo(actual.id), dist))
            else:
                arbol.agregar_nodo(Nodo(actual.id))
            # Evaluamos los vecinos en orden (se apilan para procesarse en orden DFS)
            for arista in grafo.adyacencia.get(actual.id, []):
                vecino = arista.destino
                if vecino.id not in visitados:
                    pila.append(vecino)
                    padre[vecino.id] = (actual.id, arista.distancia)
    return arbol

Pruebas de Algoritmos BFS, DFS_R y DFS_I

In [ ]:
import os

carpeta_destino = "busquedas_grafos"
os.makedirs(carpeta_destino, exist_ok=True)

grafos_creados = {
    "malla_50": (malla_50, "n_0_0"),
    "malla_200": (malla_200, "n_0_0"),
    "malla_500": (malla_500, "n_0_0"),
    "erdos_renyi_50": (erdos_renyi_50, 0),
    "erdos_renyi_200": (erdos_renyi_200, 0),
    "erdos_renyi_500": (erdos_renyi_500, 0),
    "gilbert_50": (gilbert_50, 0),
    "gilbert_200": (gilbert_200, 0),
    "gilbert_500": (gilbert_500, 0),
    "geografico_50": (geografico_50, 0),
    "geografico_200": (geografico_200, 0),
    "geografico_500": (geografico_500, 0),
    "barabasi_albert_50": (barabasi_albert_50, 0),
    "barabasi_albert_200": (barabasi_albert_200, 0),
    "barabasi_albert_500": (barabasi_albert_500, 0),
    "dorogovtsev_mendes_50": (dorogovtsev_mendes_50, 0),
    "dorogovtsev_mendes_200": (dorogovtsev_mendes_200, 0),
    "dorogovtsev_mendes_500": (dorogovtsev_mendes_500, 0)
}

for nombre, (grafo, nodo_origen) in grafos_creados.items():
    if grafo is None:
        print(f"Grafos inexistentes. Ejecutar la celda de generación anterior.")
        continue
    print(f"Procesando {nombre} desde nodo inicial '{nodo_origen}'")

    # BFS
    try:
        arbol_bfs = BFS(grafo, nodo_origen)
        arbol_bfs.guardar_graphviz(os.path.join(carpeta_destino, f"bfs_{nombre}.gv"))
    except Exception as e:
        print(f"  Error en BFS para {nombre}: {e}")

    # DFS_R (Recursivo)
    try:
        arbol_dfs_r = DFS_R(grafo, nodo_origen)
        arbol_dfs_r.guardar_graphviz(os.path.join(carpeta_destino, f"dfs_r_{nombre}.gv"))
    except Exception as e:
        print(f"  Error en DFS_R para {nombre}: {e}")

    # DFS_I (Iterativo)
    try:
        arbol_dfs_i = DFS_I(grafo, nodo_origen)
        arbol_dfs_i.guardar_graphviz(os.path.join(carpeta_destino, f"dfs_i_{nombre}.gv"))
    except Exception as e:
        print(f"  Error en DFS_I para {nombre}: {e}")

Procesando malla_50 desde nodo inicial 'n_0_0'
Procesando malla_200 desde nodo inicial 'n_0_0'
Procesando malla_500 desde nodo inicial 'n_0_0'
Procesando erdos_renyi_50 desde nodo inicial '0'
Procesando erdos_renyi_200 desde nodo inicial '0'
Procesando erdos_renyi_500 desde nodo inicial '0'
Procesando gilbert_50 desde nodo inicial '0'
Procesando gilbert_200 desde nodo inicial '0'
Procesando gilbert_500 desde nodo inicial '0'
Procesando geografico_50 desde nodo inicial '0'
Procesando geografico_200 desde nodo inicial '0'
Procesando geografico_500 desde nodo inicial '0'
Procesando barabasi_albert_50 desde nodo inicial '0'
Procesando barabasi_albert_200 desde nodo inicial '0'
Procesando barabasi_albert_500 desde nodo inicial '0'
Procesando dorogovtsev_mendes_50 desde nodo inicial '0'
Procesando dorogovtsev_mendes_200 desde nodo inicial '0'
Procesando dorogovtsev_mendes_500 desde nodo inicial '0'


In [ ]:
import shutil
from google.colab import files

# Nombre del archivo ZIP de salida
archivo_zip = "busquedas_grafos"

# Comprimir la carpeta 'busquedas_grafos'
shutil.make_archive(archivo_zip, 'zip', "busquedas_grafos")
print(f"Carpeta comprimida exitosamente como {archivo_zip}.zip")

# Descargar el archivo ZIP
files.download(f"{archivo_zip}.zip")

Carpeta comprimida exitosamente como busquedas_grafos.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Algoritmo Dijkstra

In [8]:
"""
Algoritmo de Dijkstra corregido para garantizar un árbol conexo en la exportación
"""
import heapq

def Dijkstra(grafo, s):
    # s es el ID del nodo origen
    origen = grafo._buscar_nodo_por_id(s)
    if not origen:
        raise ValueError(f"El nodo origen {s} no existe en el grafo.")

    # Inicializar distancias
    distancias = {nodo.id: float('inf') for nodo in grafo.nodos}
    distancias[s] = 0

    # Mapear predecesores en el camino más corto: hijo_id -> (padre_id, distancia_arista)
    predecesores = {}

    # Cola de prioridad para Dijkstra: (distancia_acumulada, nodo_id)
    cola_prioridad = [(0, s)]
    visitados = set()

    while cola_prioridad:
        dist_actual, u_id = heapq.heappop(cola_prioridad)

        if u_id in visitados:
            continue
        visitados.add(u_id)

        for arista in grafo.adyacencia.get(u_id, []):
            v_id = arista.destino.id
            peso = arista.distancia

            nueva_dist = dist_actual + peso
            if nueva_dist < distancias[v_id]:
                distancias[v_id] = nueva_dist
                predecesores[v_id] = (u_id, peso)
                heapq.heappush(cola_prioridad, (nueva_dist, v_id))

    # Construir el árbol de caminos más cortos
    arbol = Grafo(dirigido=True)

    # Función auxiliar para formatear la etiqueta del nodo con su distancia acumulada
    def formatear_id(n_id):
        dist = distancias[n_id]
        dist_str = f"{dist:.2f}" if dist != float('inf') else "inf"
        return f"{n_id}_dist_{dist_str}"

    # Diccionario para reutilizar siempre la misma instancia física de Nodo en el árbol
    nodos_arbol = {}
    def obtener_o_crear_nodo(n_id):
        id_formateado = formatear_id(n_id)
        if id_formateado not in nodos_arbol:
            nodos_arbol[id_formateado] = Nodo(id_formateado)
        return nodos_arbol[id_formateado]

    # Agregar nodo origen al árbol
    nodo_origen_arbol = obtener_o_crear_nodo(s)
    arbol.agregar_nodo(nodo_origen_arbol)

    # Reconstruir las conexiones óptimas garantizando nodos únicos y conectados
    for hijo_id, (padre_id, peso) in predecesores.items():
        if distancias[hijo_id] != float('inf'):
            nodo_padre = obtener_o_crear_nodo(padre_id)
            nodo_hijo = obtener_o_crear_nodo(hijo_id)
            # agregar_arista se encargará de registrar los nodos si no existían
            arbol.agregar_arista(Arista(nodo_padre, nodo_hijo, peso))

    return arbol

# Asegurar que las aristas tengan pesos reales/aleatorios para Dijkstra (excepto el geografico que ya usa distancias euclidianas)
def asignar_pesos_aleatorios(grafo):
    for arista in grafo.get_aristas():
        # Asignar un peso aleatorio entre 1.0 y 15.0 redondeado a 2 decimales
        arista.distancia = round(random.uniform(1.0, 15.0), 2)
    # Reconstruir adyacencia para mantener consistencia de las referencias de pesos
    for nodo_id, lista_aristas in grafo.adyacencia.items():
        for arista in lista_aristas:
            # Buscar la arista correspondiente en el set para sincronizar el peso cambiado
            for a_original in grafo.aristas:
                if (a_original.origen.id == arista.origen.id and a_original.destino.id == arista.destino.id) or (not grafo.dirigido and a_original.origen.id == arista.destino.id and a_original.destino.id == arista.origen.id):
                    arista.distancia = a_original.distancia

Generar grafos con pocos y muchos nodos para **Dijkstra**

In [9]:
import os
import random
import shutil
from google.colab import files

# 1. Definir directorios de destino independientes
carpeta_originales = "grafos_parte3"
carpeta_dijkstra = "dijkstra_grafos"

os.makedirs(carpeta_originales, exist_ok=True)
os.makedirs(carpeta_dijkstra, exist_ok=True)

# Generar conjuntos con pocos (50) y muchos (300) nodos
grafos_ejemplos = {
    "malla_pocos": (grafoMalla(5, 10), "n_0_0"),
    "malla_muchos": (grafoMalla(15, 20), "n_0_0"),
    "erdos_pocos": (grafoErdosRenyi(50, 120), 0),
    "erdos_muchos": (grafoErdosRenyi(300, 800), 0),
    "gilbert_pocos": (grafoGilbert(50, 0.1), 0),
    "gilbert_muchos": (grafoGilbert(300, 0.03), 0),
    "geografico_pocos": (grafoGeografico(50, 0.25), 0),
    "geografico_muchos": (grafoGeografico(300, 0.1), 0),
    "barabasi_pocos": (grafoBarabasiAlbert(50, 3), 0),
    "barabasi_muchos": (grafoBarabasiAlbert(300, 4), 0),
    "dorogovtsev_pocos": (grafoDorogovtsevMendes(50), 0),
    "dorogovtsev_muchos": (grafoDorogovtsevMendes(300), 0)
}

# Procesar asignación de pesos, guardado original y cálculo de Dijkstra
for nombre, (grafo, nodo_origen) in grafos_ejemplos.items():
    # Asignar pesos aleatorios (al modelo geográfico no, ya que usa sus distancias euclidianas)
    if "geografico" not in nombre:
        asignar_pesos_aleatorios(grafo)

    print(f"Procesando {nombre}...")

    # Guardar grafo original con pesos en la primera carpeta
    ruta_original = os.path.join(carpeta_originales, f"original_{nombre}.gv")
    grafo.guardar_graphviz(ruta_original)

    # Calcular Dijkstra y guardar en la segunda carpeta
    try:
        arbol_dijkstra = Dijkstra(grafo, nodo_origen)
        ruta_dijkstra = os.path.join(carpeta_dijkstra, f"dijkstra_{nombre}.gv")
        arbol_dijkstra.guardar_graphviz(ruta_dijkstra)
    except Exception as e:
        print(f"  Error en Dijkstra para {nombre}: {e}")

# Comprimir y descargar la carpeta de grafos originales (antes de Dijkstra)
shutil.make_archive(carpeta_originales, 'zip', carpeta_originales)
print(f"\nArchivo {carpeta_originales}.zip listo para descargar.")
files.download(f"{carpeta_originales}.zip")

# Comprimir y descargar la carpeta de resultados de Dijkstra (después de Dijkstra)
shutil.make_archive(carpeta_dijkstra, 'zip', carpeta_dijkstra)
print(f"Archivo {carpeta_dijkstra}.zip listo para descargar.")
files.download(f"{carpeta_dijkstra}.zip")

Procesando malla_pocos...
Procesando malla_muchos...
Procesando erdos_pocos...
Procesando erdos_muchos...
Procesando gilbert_pocos...
Procesando gilbert_muchos...
Procesando geografico_pocos...
Procesando geografico_muchos...
Procesando barabasi_pocos...
Procesando barabasi_muchos...
Procesando dorogovtsev_pocos...
Procesando dorogovtsev_muchos...

Archivo grafos_parte3.zip listo para descargar.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archivo dijkstra_grafos.zip listo para descargar.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>